In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#linear regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
#k-nearest neighbors regressor
from sklearn.neighbors import KNeighborsRegressor
#random forest regressor
from sklearn.ensemble import RandomForestRegressor
# evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error, mean_absolute_error
#gradient boosting regressor
from sklearn.ensemble import GradientBoostingRegressor
# pipelines
from sklearn.pipeline import Pipeline
#scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
df = pd.read_csv("formatted_gdp_growth_data.csv")
df.isna().sum()

year                                    0
gdp_growth(%)                           0
gdp_growth_lag1(%)                      1
gdp_growth_lag2(%)                      2
household_consumption(%)                0
unemployment_rate(%)                   11
under_earning_percentage(%)            13
inflation_rate(%)                       0
lending_interest_rate(%)_x              6
fdi_net_flows(US$)                      0
tax_revenue_percentage(%)              18
state_revenues_percentage(%)           18
gross_capital_formation_growth(%)_x     0
import_of_goods_and_services(US$)       0
export_of_goods_and_services(US$)       0
gross_capital_formation_growth(%)_y     0
exchange_rate(IDR/US$)                  0
dtype: int64

In [3]:
df.describe()

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
count,44.000000,44.000000,43.000000,42.000000,44.000000,33.00000,31.000000,44.000000,38.000000,4.400000e+01,26.000000,26.000000,44.000000,4.400000e+01,4.400000e+01,44.000000,44.000000
mean,2001.500000,4.955050,4.952864,4.944428,70.105728,5.06400,41.058065,8.753567,16.540965,7.589957e+09,15.860978,19.097463,24.451539,1.006864e+11,1.083679e+11,24.451539,7213.774407
std,12.845233,3.425071,3.465575,3.507137,4.371609,1.53784,25.570983,8.606848,5.683718,9.445301e+09,2.977557,2.298816,123.922857,8.409564e+10,8.827531e+10,123.922857,5086.558893
min,1980.000000,-13.126725,-13.126725,-13.126725,60.734725,2.61700,6.700000,1.560079,8.520000,-4.550355e+09,11.057787,15.108254,-164.509353,1.607649e+10,1.638707e+10,-164.509353,626.994000
25%,1990.750000,4.760247,4.740124,4.720001,67.084308,4.04900,19.800000,4.640005,12.017083,2.835000e+08,14.265207,17.412376,-9.587040,2.995771e+10,3.204349e+10,-9.587040,1923.441458
50%,2001.500000,5.240744,5.307197,5.337622,69.665130,4.61100,38.000000,6.586427,15.238750,3.227500e+09,15.516179,18.506774,5.652888,5.799963e+10,6.578898e+10,5.652888,8854.641667
75%,2012.250000,6.382869,6.420715,6.458562,72.436718,6.10600,64.000000,9.810600,20.765417,1.897614e+10,17.410716,20.653019,30.929111,1.728422e+11,1.834971e+11,30.929111,10407.763125
max,2023.000000,9.880078,9.880078,9.880078,81.334706,8.06000,86.800000,58.451041,32.154167,2.512073e+10,21.946434,25.474300,627.499564,2.765068e+11,3.232182e+11,627.499564,15236.884662


In [4]:
df[0:40]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
0,1980,9.880078,NaN,NaN,70.834204,NaN,NaN,18.035422,NaN,1.800000e+08,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,24.369558,626.994000
1,1981,7.927157,9.880078,NaN,76.531919,NaN,NaN,12.265908,NaN,1.330000e+08,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,12.398103,631.756667
2,1982,2.246445,7.927157,9.880078,81.334706,NaN,NaN,9.445425,NaN,2.250000e+08,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,5.647824,661.420750
3,1983,4.192967,2.246445,7.927157,71.666649,NaN,NaN,11.799738,NaN,2.920000e+08,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,627.499564,909.264833
4,1984,6.975528,4.192967,2.246445,72.352702,NaN,86.2,10.455039,NaN,2.220000e+08,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,52.568955,1025.944833
5,1985,2.462144,6.975528,4.192967,72.424958,NaN,NaN,4.724535,NaN,3.100000e+08,18.750897,21.481026,46.486894,1.786022e+10,2.027985e+10,46.486894,1110.580000
6,1986,5.875045,2.462144,6.975528,75.208760,NaN,NaN,5.822666,21.504167,2.580000e+08,14.620770,20.794591,-3.995642,1.640173e+10,1.638707e+10,-3.995642,1282.560000
7,1987,4.925927,5.875045,2.462144,71.411564,NaN,86.8,9.278654,21.666667,3.850000e+08,15.083695,19.853882,-18.896794,1.700630e+10,1.866107e+10,-18.896794,1643.848333
8,1988,5.780498,4.925927,5.875045,72.114454,NaN,NaN,8.045357,22.097500,5.760000e+08,15.083938,16.950870,-71.621388,1.872552e+10,2.111016e+10,-71.621388,1685.704167
9,1989,7.456587,5.780498,4.925927,69.343427,NaN,NaN,6.415541,21.699167,6.820000e+08,15.957202,17.401712,22.952061,2.171847e+10,2.464007e+10,22.952061,1770.059167


In [5]:
# impute gdp_growth lag 1 and 2 with linear regression
x = df[["fdi_net_flows(US$)","gross_capital_formation_growth(%)_y", "exchange_rate(IDR/US$)"]]
x = x.iloc[1:40]

y = df["gdp_growth_lag1(%)"]
y = y.iloc[1:40]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

model_impute_lag1 = Lasso(alpha=0.1)
model_impute_lag1.fit(x_train, y_train)

,alpha,0.1
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [6]:
# test the model performance on the test set
np.random.seed(42)
y_pred = model_impute_lag1.predict(x_test)

performance = {
    "MSE": mean_squared_error(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
    "RMSE" : root_mean_squared_error(y_test, y_pred)
}

print(performance)


{'MSE': 4.927428540573947, 'MAE': 1.7992104514695815, 'R2': -2.1201289271728343, 'RMSE': 2.21978119204888}


In [7]:
# use the model to predict the lag1 missing value
missing_lag1 = df[df["gdp_growth_lag1(%)"].isna()]
missing_lag1_x = missing_lag1[["fdi_net_flows(US$)","gross_capital_formation_growth(%)_y", "exchange_rate(IDR/US$)"]]
predicted_lag1 = model_impute_lag1.predict(missing_lag1_x)
df.loc[df["gdp_growth_lag1(%)"].isna(), "gdp_growth_lag1(%)"] = predicted_lag1

In [8]:
df.head()

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
0,1980,9.880078,6.735124,NaN,70.834204,NaN,NaN,18.035422,NaN,180000000.0,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,24.369558,626.994000
1,1981,7.927157,9.880078,NaN,76.531919,NaN,NaN,12.265908,NaN,133000000.0,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,12.398103,631.756667
2,1982,2.246445,7.927157,9.880078,81.334706,NaN,NaN,9.445425,NaN,225000000.0,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,5.647824,661.420750
3,1983,4.192967,2.246445,7.927157,71.666649,NaN,NaN,11.799738,NaN,292000000.0,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,627.499564,909.264833
4,1984,6.975528,4.192967,2.246445,72.352702,NaN,86.2,10.455039,NaN,222000000.0,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,52.568955,1025.944833


# Impute lag_2 with Lasso

In [9]:
x = df[["fdi_net_flows(US$)","gross_capital_formation_growth(%)_y", "exchange_rate(IDR/US$)"]]
x = x.iloc[2:40]

y = df["gdp_growth_lag2(%)"]
y = y.iloc[2:40]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# create a pipeline for the model with normalization and lasso regression       
pipeline = Pipeline([
    ("normalization", MinMaxScaler()),
    ("model", Ridge(alpha=0.1))
])
pipeline.fit(x_train, y_train)


,steps,"[('normalization', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,feature_range,"(0, ...)"
,copy,True
,clip,False
,alpha,0.1
,fit_intercept,True
,copy_X,True
,max_iter,None


In [10]:
# evaluate the model performance on the test set
y_pred = pipeline.predict(x_test)
performance = {
    "MSE": mean_squared_error(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
    "RMSE" : root_mean_squared_error(y_test, y_pred)
}
print(performance)

{'MSE': 1.5936181716843048, 'MAE': 0.874695886106394, 'R2': -3.6144120784447518, 'RMSE': 1.2623859044223777}


In [11]:
# use the model to impute the missing values in gdp_growth_lag2

missing_lag2 = df[df["gdp_growth_lag2(%)"].isna()]
missing_lag2_x = missing_lag2[["fdi_net_flows(US$)","gross_capital_formation_growth(%)_y", "exchange_rate(IDR/US$)"]]
predicted_lag2 = pipeline.predict(missing_lag2_x)
df.loc[df["gdp_growth_lag2(%)"].isna(), "gdp_growth_lag2(%)"] = predicted_lag2

In [12]:

# fix lag 1
df["gdp_growth_lag1(%)"].iloc[0] = df["gdp_growth_lag2(%)"].iloc[1]
df.head()

/var/folders/tk/mn4_r_cn03ndtjss8b670dfw0000gn/T/ipykernel_4406/2192646626.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["gdp_growth_lag1(%)"].iloc[0] = df["gdp_growth_lag2(%)"].iloc[1]
/var/folders/tk/mn4_r_cn03ndtjss8b670dfw0000gn

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
0,1980,9.880078,6.165775,6.163400,70.834204,NaN,NaN,18.035422,NaN,180000000.0,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,24.369558,626.994000
1,1981,7.927157,9.880078,6.165775,76.531919,NaN,NaN,12.265908,NaN,133000000.0,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,12.398103,631.756667
2,1982,2.246445,7.927157,9.880078,81.334706,NaN,NaN,9.445425,NaN,225000000.0,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,5.647824,661.420750
3,1983,4.192967,2.246445,7.927157,71.666649,NaN,NaN,11.799738,NaN,292000000.0,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,627.499564,909.264833
4,1984,6.975528,4.192967,2.246445,72.352702,NaN,86.2,10.455039,NaN,222000000.0,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,52.568955,1025.944833


# Impute Unemployment Rate with Regression model

In [13]:
# split data
x = df[["gdp_growth_lag1(%)", "gdp_growth_lag2(%)"]]
x = x.iloc[18:40]

y = df["unemployment_rate(%)"]
y = y.iloc[18:40]

# train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# experiment models  
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.1),
}

performance = {
    "MAE" : [],
    "MSE" : [], 
    "RMSE" : [],
    "R2" : []
}

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    performance["MAE"].append(mean_absolute_error(y_test, y_pred))
    performance["MSE"].append(mean_squared_error(y_test, y_pred))
    performance["RMSE"].append(root_mean_squared_error(y_test, y_pred))
    performance["R2"].append(r2_score(y_test, y_pred))

    # format into pandas data
performance_df = pd.DataFrame(performance, index=models.keys())
print(performance_df)


                       MAE       MSE      RMSE        R2
LinearRegression  1.030005  1.479729  1.216441 -0.222621
Ridge             1.027041  1.471568  1.213082 -0.215878
Lasso             0.899254  1.199115  1.095041  0.009235


In [14]:
# impute the missing values in the unemployment_rate column with the best model (Lasso)
np.random.seed(42)
Lasso_model_ = Lasso(alpha=0.1)
Lasso_model_.fit(x_train, y_train)
x_imputed = df[df["unemployment_rate(%)"].isna()][["gdp_growth_lag1(%)", "gdp_growth_lag2(%)"]]
y_pred_imputed = Lasso_model_.predict(x_imputed)
df.loc[df["unemployment_rate(%)"].isna(), "unemployment_rate(%)"] = y_pred_imputed

# Under-erning percentage

In [15]:
np.random.seed(42)
# split data
x = df[["gdp_growth_lag1(%)", "gdp_growth_lag2(%)", "unemployment_rate(%)", 
        "inflation_rate(%)", "exchange_rate(IDR/US$)", "household_consumption(%)"]]
x = x.iloc[18:40]

y = df["under_earning_percentage(%)"]
y = y.iloc[18:40]

# train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# experiment models  
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.1),
}

performance = {
    "MAE" : [],
    "MSE" : [], 
    "RMSE" : [],
    "R2" : []
}

for name, model in models.items():
    moodel = Pipeline([
        ("normalization", StandardScaler()),
        ("model", model)
    ])
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    performance["MAE"].append(mean_absolute_error(y_test, y_pred))
    performance["MSE"].append(mean_squared_error(y_test, y_pred))
    performance["RMSE"].append(root_mean_squared_error(y_test, y_pred))
    performance["R2"].append(r2_score(y_test, y_pred))

# format into pandas data frame
performance_df = pd.DataFrame(performance, index=models.keys())
print(performance_df)

                        MAE         MSE       RMSE        R2
LinearRegression  16.924452  472.636245  21.740199 -0.065325
Ridge             16.706921  459.770745  21.442265 -0.036326
Lasso             15.470655  395.657492  19.891141  0.108185


In [16]:
# impute the missing values in the unemployment_rate column with the best model (Lasso)
np.random.seed(42)
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(x_train, y_train)
x_imputed = df[df["under_earning_percentage(%)"].isna()][["gdp_growth_lag1(%)", "gdp_growth_lag2(%)", "unemployment_rate(%)", 
        "inflation_rate(%)", "exchange_rate(IDR/US$)", "household_consumption(%)"]]
y_pred_imputed = lasso_model.predict(x_imputed)
df.loc[df["under_earning_percentage(%)"].isna(), "under_earning_percentage(%)"] = y_pred_imputed

In [17]:
df[0:45]

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
0,1980,9.880078,6.165775,6.163400,70.834204,5.620902,89.164321,18.035422,NaN,1.800000e+08,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,24.369558,626.994000
1,1981,7.927157,9.880078,6.165775,76.531919,5.620827,88.921829,12.265908,NaN,1.330000e+08,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,12.398103,631.756667
2,1982,2.246445,7.927157,9.880078,81.334706,5.504006,75.150527,9.445425,NaN,2.250000e+08,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,5.647824,661.420750
3,1983,4.192967,2.246445,7.927157,71.666649,5.565429,67.419547,11.799738,NaN,2.920000e+08,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,627.499564,909.264833
4,1984,6.975528,4.192967,2.246445,72.352702,5.744096,86.200000,10.455039,NaN,2.220000e+08,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,52.568955,1025.944833
5,1985,2.462144,6.975528,4.192967,72.424958,5.682875,71.527427,4.724535,NaN,3.100000e+08,18.750897,21.481026,46.486894,1.786022e+10,2.027985e+10,46.486894,1110.580000
6,1986,5.875045,2.462144,6.975528,75.208760,5.595359,60.273201,5.822666,21.504167,2.580000e+08,14.620770,20.794591,-3.995642,1.640173e+10,1.638707e+10,-3.995642,1282.560000
7,1987,4.925927,5.875045,2.462144,71.411564,5.737312,86.800000,9.278654,21.666667,3.850000e+08,15.083695,19.853882,-18.896794,1.700630e+10,1.866107e+10,-18.896794,1643.848333
8,1988,5.780498,4.925927,5.875045,72.114454,5.629971,67.660828,8.045357,22.097500,5.760000e+08,15.083938,16.950870,-71.621388,1.872552e+10,2.111016e+10,-71.621388,1685.704167
9,1989,7.456587,5.780498,4.925927,69.343427,5.659822,66.733250,6.415541,21.699167,6.820000e+08,15.957202,17.401712,22.952061,2.171847e+10,2.464007e+10,22.952061,1770.059167


# Interpolate Tax Revenue 

In [18]:
# impute the missing values in tax_revenue_percentage with linear interpolation
df["tax_revenue_percentage(%)"].iloc[0:30] = df["tax_revenue_percentage(%)"].iloc[0:30].interpolate(method="linear")
df[0:45]

/var/folders/tk/mn4_r_cn03ndtjss8b670dfw0000gn/T/ipykernel_4406/1948936870.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["tax_revenue_percentage(%)"].iloc[0:30] = df["tax_revenue_percentage(%)"].iloc[0:30].interpolate(method="linear

,year,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%)_x,fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%)_x,import_of_goods_and_services(US$),export_of_goods_and_services(US$),gross_capital_formation_growth(%)_y,exchange_rate(IDR/US$)
0,1980,9.880078,6.165775,6.163400,70.834204,5.620902,89.164321,18.035422,NaN,1.800000e+08,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,24.369558,626.994000
1,1981,7.927157,9.880078,6.165775,76.531919,5.620827,88.921829,12.265908,NaN,1.330000e+08,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,12.398103,631.756667
2,1982,2.246445,7.927157,9.880078,81.334706,5.504006,75.150527,9.445425,NaN,2.250000e+08,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,5.647824,661.420750
3,1983,4.192967,2.246445,7.927157,71.666649,5.565429,67.419547,11.799738,NaN,2.920000e+08,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,627.499564,909.264833
4,1984,6.975528,4.192967,2.246445,72.352702,5.744096,86.200000,10.455039,NaN,2.220000e+08,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,52.568955,1025.944833
5,1985,2.462144,6.975528,4.192967,72.424958,5.682875,71.527427,4.724535,NaN,3.100000e+08,18.750897,21.481026,46.486894,1.786022e+10,2.027985e+10,46.486894,1110.580000
6,1986,5.875045,2.462144,6.975528,75.208760,5.595359,60.273201,5.822666,21.504167,2.580000e+08,14.620770,20.794591,-3.995642,1.640173e+10,1.638707e+10,-3.995642,1282.560000
7,1987,4.925927,5.875045,2.462144,71.411564,5.737312,86.800000,9.278654,21.666667,3.850000e+08,15.083695,19.853882,-18.896794,1.700630e+10,1.866107e+10,-18.896794,1643.848333
8,1988,5.780498,4.925927,5.875045,72.114454,5.629971,67.660828,8.045357,22.097500,5.760000e+08,15.083938,16.950870,-71.621388,1.872552e+10,2.111016e+10,-71.621388,1685.704167
9,1989,7.456587,5.780498,4.925927,69.343427,5.659822,66.733250,6.415541,21.699167,6.820000e+08,15.957202,17.401712,22.952061,2.171847e+10,2.464007e+10,22.952061,1770.059167


# Fill Tax Revenue[30:44] missing value with regression model

In [19]:
# split data
x = df[["gdp_growth(%)", "unemployment_rate(%)", 
        "inflation_rate(%)", "exchange_rate(IDR/US$)", "household_consumption(%)", "under_earning_percentage(%)"]]
x = x.iloc[0:30]

y = df["tax_revenue_percentage(%)"]
y = y.iloc[0:30]

# train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=42)

# experiment models
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.1)}


performance = {
    "MAE" : [],
    "MSE" : [],
    "RMSE" : [],
    "R2" : []
}
np.random.seed(42)

for name, model in models.items():
    pipeline = Pipeline([
        ("normalization", (MinMaxScaler())),
        ("model", model)
    ])
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    performance["MAE"].append(mean_absolute_error(y_test, y_pred))
    performance["MSE"].append(mean_squared_error(y_test, y_pred))
    performance["RMSE"].append(root_mean_squared_error(y_test, y_pred))
    performance["R2"].append(r2_score(y_test, y_pred))
# format into pandas data frame
performance_df = pd.DataFrame(performance, index=models.keys())
print(performance_df)

                       MAE       MSE      RMSE        R2
LinearRegression  1.287522  1.965662  1.402021 -0.071127
Ridge             1.236588  1.782749  1.335196  0.028546
Lasso             1.101108  1.523620  1.234350  0.169750


In [20]:
# fill the missing values in tax_revenue_percentage with the best model (Lasso)
pipeline = Pipeline([
    ("normalization", (MinMaxScaler())),
    ("model", Lasso(alpha=0.1))
])
pipeline.fit(x_train, y_train)
x_imputed = df[df["tax_revenue_percentage(%)"].isna()][["gdp_growth(%)", "unemployment_rate(%)", 
        "inflation_rate(%)", "exchange_rate(IDR/US$)", "household_consumption(%)", "under_earning_percentage(%)"]]
y_pred_imputed = pipeline.predict(x_imputed)
df.loc[df["tax_revenue_percentage(%)"].isna(), "tax_revenue_percentage(%)"] = y_pred_imputed

In [21]:
df["tax_revenue_percentage(%)"][30:45]

30    11.477271
31    11.359080
32    11.017332
33    10.719321
34    10.124576
35     9.396496
36     9.256591
37     8.945110
38     8.446254
39     8.479459
40     8.480796
41     8.078380
42     7.336426
43     7.261091
Name: tax_revenue_percentage(%), dtype: float64

# Interpolate State Revenue Percentage

In [22]:
df["state_revenues_percentage(%)"] = df["state_revenues_percentage(%)"].interpolate(method="linear", order=2)

In [23]:
df["state_revenues_percentage(%)"][30:45]

30    15.108254
31    15.108254
32    15.108254
33    15.108254
34    15.108254
35    15.108254
36    15.108254
37    15.108254
38    15.108254
39    15.108254
40    15.108254
41    15.108254
42    15.108254
43    15.108254
Name: state_revenues_percentage(%), dtype: float64

# Impute Lending interest rate

In [24]:
df.isna().sum()

year                                   0
gdp_growth(%)                          0
gdp_growth_lag1(%)                     0
gdp_growth_lag2(%)                     0
household_consumption(%)               0
unemployment_rate(%)                   0
under_earning_percentage(%)            0
inflation_rate(%)                      0
lending_interest_rate(%)_x             6
fdi_net_flows(US$)                     0
tax_revenue_percentage(%)              0
state_revenues_percentage(%)           0
gross_capital_formation_growth(%)_x    0
import_of_goods_and_services(US$)      0
export_of_goods_and_services(US$)      0
gross_capital_formation_growth(%)_y    0
exchange_rate(IDR/US$)                 0
dtype: int64

In [25]:
df["lending_interest_rate(%)_x"].head(20)

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
5           NaN
6     21.504167
7     21.666667
8     22.097500
9     21.699167
10    20.825000
11    25.533333
12    24.033333
13    20.586667
14    17.760000
15    18.851667
16    19.217500
17    21.817500
18    32.154167
19    27.662500
Name: lending_interest_rate(%)_x, dtype: float64

In [26]:
# backfill
df["lending_interest_rate(%)_x"] = df["lending_interest_rate(%)_x"].fillna(method="bfill")
df["lending_interest_rate(%)_x"].head(20)

/var/folders/tk/mn4_r_cn03ndtjss8b670dfw0000gn/T/ipykernel_4406/2311269517.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["lending_interest_rate(%)_x"] = df["lending_interest_rate(%)_x"].fillna(method="bfill")


0     21.504167
1     21.504167
2     21.504167
3     21.504167
4     21.504167
5     21.504167
6     21.504167
7     21.666667
8     22.097500
9     21.699167
10    20.825000
11    25.533333
12    24.033333
13    20.586667
14    17.760000
15    18.851667
16    19.217500
17    21.817500
18    32.154167
19    27.662500
Name: lending_interest_rate(%)_x, dtype: float64

# Remove duplicate and format the label in a proper way

In [27]:
df.columns

Index(['year', 'gdp_growth(%)', 'gdp_growth_lag1(%)', 'gdp_growth_lag2(%)',
       'household_consumption(%)', 'unemployment_rate(%)',
       'under_earning_percentage(%)', 'inflation_rate(%)',
       'lending_interest_rate(%)_x', 'fdi_net_flows(US$)',
       'tax_revenue_percentage(%)', 'state_revenues_percentage(%)',
       'gross_capital_formation_growth(%)_x',
       'import_of_goods_and_services(US$)',
       'export_of_goods_and_services(US$)',
       'gross_capital_formation_growth(%)_y', 'exchange_rate(IDR/US$)'],
      dtype='object')

In [28]:
# change the column name of lending_interest_rate(%)_x to lending_interest_rate(%)
df.rename(columns={"lending_interest_rate(%)_x": "lending_interest_rate(%)"}, inplace=True)
# change the column name of lending_interest_rate(%)_y to lending_interest_rate(%)
df.rename(columns={"lending_interest_rate(%)_y": "lending_interest_rate(%)"}, inplace=True)
#change the column name of state_revenues_percentage(%) to state_revenues_percentage(%)
df.rename(columns={"state_revenues_percentage(%)": "state_revenues_percentage(%)"}, inplace=True)
#change the colun name of gross_capital_formation_growth(%)_x to gross_capital_formation_growth(%)
df.rename(columns={"gross_capital_formation_growth(%)_x": "gross_capital_formation_growth(%)"}, inplace=True)
# remove the column gross_capital_formation_growth(%)_y
df.drop(columns=["gross_capital_formation_growth(%)_y"], inplace=True)

In [29]:
df.columns

Index(['year', 'gdp_growth(%)', 'gdp_growth_lag1(%)', 'gdp_growth_lag2(%)',
       'household_consumption(%)', 'unemployment_rate(%)',
       'under_earning_percentage(%)', 'inflation_rate(%)',
       'lending_interest_rate(%)', 'fdi_net_flows(US$)',
       'tax_revenue_percentage(%)', 'state_revenues_percentage(%)',
       'gross_capital_formation_growth(%)',
       'import_of_goods_and_services(US$)',
       'export_of_goods_and_services(US$)', 'exchange_rate(IDR/US$)'],
      dtype='object')

In [30]:
df.columns.duplicated()

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False])

In [31]:
# load the cleaned data into a new csv file
df.to_csv("cleaned_gdp_growth_data.csv", index=False)

# Create Lag-1 Data set

In [32]:
# Set 'year' as the index so it's preserved during shifts
df_indexed = df.set_index('year')

#Define your target (the original values we want to predict)
target_cols = ['gdp_growth(%)', 'gdp_growth_lag1(%)', 'gdp_growth_lag2(%)']
target = df_indexed[target_cols]

#Define your features and Shift them by 1
features = df_indexed.drop(columns=target_cols)
features_lag1 = features.shift(1)

# Combine them
df_lag1 = target.join(features_lag1).dropna()


In [33]:
df_lag1

,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%),unemployment_rate(%),under_earning_percentage(%),inflation_rate(%),lending_interest_rate(%),fdi_net_flows(US$),tax_revenue_percentage(%),state_revenues_percentage(%),gross_capital_formation_growth(%),import_of_goods_and_services(US$),export_of_goods_and_services(US$),exchange_rate(IDR/US$)
year,,,,,,,,,,,,,,,
1981,7.927157,9.880078,6.165775,70.834204,5.620902,89.164321,18.035422,21.504167,1.800000e+08,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,626.994000
1982,2.246445,7.927157,9.880078,76.531919,5.620827,88.921829,12.265908,21.504167,1.330000e+08,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,631.756667
1983,4.192967,2.246445,7.927157,81.334706,5.504006,75.150527,9.445425,21.504167,2.250000e+08,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,661.420750
1984,6.975528,4.192967,2.246445,71.666649,5.565429,67.419547,11.799738,21.504167,2.920000e+08,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,909.264833
1985,2.462144,6.975528,4.192967,72.352702,5.744096,86.200000,10.455039,21.504167,2.220000e+08,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,1025.944833
1986,5.875045,2.462144,6.975528,72.424958,5.682875,71.527427,4.724535,21.504167,3.100000e+08,18.750897,21.481026,46.486894,1.786022e+10,2.027985e+10,1110.580000
1987,4.925927,5.875045,2.462144,75.208760,5.595359,60.273201,5.822666,21.504167,2.580000e+08,14.620770,20.794591,-3.995642,1.640173e+10,1.638707e+10,1282.560000
1988,5.780498,4.925927,5.875045,71.411564,5.737312,86.800000,9.278654,21.666667,3.850000e+08,15.083695,19.853882,-18.896794,1.700630e+10,1.866107e+10,1643.848333
1989,7.456587,5.780498,4.925927,72.114454,5.629971,67.660828,8.045357,22.097500,5.760000e+08,15.083938,16.950870,-71.621388,1.872552e+10,2.111016e+10,1685.704167


In [34]:
df_lag1.to_csv("cleaned_gdp_growth_data_lag1.csv", index=False)

In [35]:
# change the name
labels = df_lag1.columns[3:]

for label in labels:
    df_lag1.rename(columns={label: label + "_lag1"}, inplace=True)
    

In [36]:
df_lag1

,gdp_growth(%),gdp_growth_lag1(%),gdp_growth_lag2(%),household_consumption(%)_lag1,unemployment_rate(%)_lag1,under_earning_percentage(%)_lag1,inflation_rate(%)_lag1,lending_interest_rate(%)_lag1,fdi_net_flows(US$)_lag1,tax_revenue_percentage(%)_lag1,state_revenues_percentage(%)_lag1,gross_capital_formation_growth(%)_lag1,import_of_goods_and_services(US$)_lag1,export_of_goods_and_services(US$)_lag1,exchange_rate(IDR/US$)_lag1
year,,,,,,,,,,,,,,,
1981,7.927157,9.880078,6.165775,70.834204,5.620902,89.164321,18.035422,21.504167,1.800000e+08,21.779838,22.897656,24.369558,1.607649e+10,2.208839e+10,626.994000
1982,2.246445,7.927157,9.880078,76.531919,5.620827,88.921829,12.265908,21.504167,1.330000e+08,21.946434,25.474300,12.398103,2.184722e+10,2.362907e+10,631.756667
1983,4.192967,2.246445,7.927157,81.334706,5.504006,75.150527,9.445425,21.504167,2.250000e+08,20.056144,21.489923,5.647824,2.370914e+10,2.017659e+10,661.420750
1984,6.975528,4.192967,2.246445,71.666649,5.565429,67.419547,11.799738,21.504167,2.920000e+08,18.822865,21.046818,627.499564,2.335427e+10,2.248829e+10,909.264833
1985,2.462144,6.975528,4.192967,72.352702,5.744096,86.200000,10.455039,21.504167,2.220000e+08,17.484389,21.508291,52.568955,1.934295e+10,2.317777e+10,1025.944833
1986,5.875045,2.462144,6.975528,72.424958,5.682875,71.527427,4.724535,21.504167,3.100000e+08,18.750897,21.481026,46.486894,1.786022e+10,2.027985e+10,1110.580000
1987,4.925927,5.875045,2.462144,75.208760,5.595359,60.273201,5.822666,21.504167,2.580000e+08,14.620770,20.794591,-3.995642,1.640173e+10,1.638707e+10,1282.560000
1988,5.780498,4.925927,5.875045,71.411564,5.737312,86.800000,9.278654,21.666667,3.850000e+08,15.083695,19.853882,-18.896794,1.700630e+10,1.866107e+10,1643.848333
1989,7.456587,5.780498,4.925927,72.114454,5.629971,67.660828,8.045357,22.097500,5.760000e+08,15.083938,16.950870,-71.621388,1.872552e+10,2.111016e+10,1685.704167


In [40]:
df_lag1.to_csv("cleaned_gdp_growth_lag1.csv", index=True)